In [0]:
%run ../config_setup

In [0]:
# Databricks Notebook: sim_04_gps (parameterized)
# Schedule: Every 30 min

import uuid, random, datetime as dt
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
ROOT = LANDING
SEED = f"{ROOT}/_seed"

try:
    machines  = [r.asDict() for r in spark.read.json(f"{SEED}/machines").collect()]
    sites     = [r.asDict() for r in spark.read.json(f"{SEED}/sites").collect()]
    geofences = [r.asDict() for r in spark.read.json(f"{SEED}/geofences").collect()]
    if not machines: raise Exception()
    print(f"✅ Seed: {len(machines)} machines, {len(sites)} sites, {len(geofences)} geofences")
except:
    dbutils.notebook.exit("❌ SEED MISSING — run sim_00_seed first")

machine_ids = [m["machine_id"] for m in machines]
site_ids    = [s["site_id"] for s in sites]
geofence_ids= [g["geofence_id"] for g in geofences]
site_by_id  = {s["site_id"]: s for s in sites}

run_ts = dt.datetime.utcnow()
run_id = f"{run_ts:%Y%m%dT%H%M%S}_{uuid.uuid4().hex[:6]}"

# ---- location_ping (FK: machine_id, near site's geo) ----
pings = []
for _ in range(random.randint(50, 150)):
    mid = random.choice(machine_ids)
    sid = random.choice(site_ids)
    s = site_by_id[sid]
    pings.append({
        "ping_id": f"GPS-{uuid.uuid4().hex[:8]}",
        "machine_id": mid,
        "site_id": sid,
        "ping_ts": run_ts.isoformat()+"Z",
        "lat": round(s["geo_lat"] + random.uniform(-0.01, 0.01), 6),
        "lon": round(s["geo_lon"] + random.uniform(-0.01, 0.01), 6),
        "speed": round(random.uniform(0, 40), 1),
    })
spark.createDataFrame(pings).coalesce(1).write.mode("append") \
    .option("header","true").csv(f"{ROOT}/gps/location_ping/{run_id}.csv")
print(f"  ✓ location_ping: {len(pings)}")

# ---- geofence_event (FK: machine_id, geofence_id) ----
geo_events = [{
    "geo_event_id": f"GE-{uuid.uuid4().hex[:8]}",
    "machine_id": random.choice(machine_ids),
    "geofence_id": random.choice(geofence_ids),
    "enter_ts": run_ts.isoformat()+"Z",
    "exit_ts": (run_ts + dt.timedelta(minutes=random.randint(5, 60))).isoformat()+"Z",
} for _ in range(random.randint(3, 10))]
spark.createDataFrame(geo_events).coalesce(1).write.mode("append") \
    .option("header","true").csv(f"{ROOT}/gps/geofence_event/{run_id}.csv")
print(f"  ✓ geofence_event: {len(geo_events)}")

# ---- geofence_master (FK: site_id) — only initial load ----
try:
    spark.read.json(f"{ROOT}/gps/geofence_master").count()
    print("  ⏭  geofence_master exists — skipping (static)")
except:
    gm = [{
        "geofence_id": g["geofence_id"],
        "site_id": g["site_id"],
        "geofence_name": g["geofence_name"],
        "boundary_wkt": "POLYGON((0 0,1 0,1 1,0 1,0 0))",
    } for g in geofences]
    spark.createDataFrame(gm).coalesce(1).write.mode("overwrite") \
        .option("header","true").csv(f"{ROOT}/gps/geofence_master/")
    print(f"  ✓ geofence_master: {len(gm)} (initial load)")

print(f"\n✅ GPS run complete: {run_id}")